<a href="https://colab.research.google.com/github/tanjun8802/Mase_EDGE/blob/jtan%2Fdev/EDGE_NAS_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mase_EDGE - Optuna Study and Executorch export Pipeline

Mase EDGE performs Optuna study for a specific Android phone that connects to the host PC. Alongside optimisation, the Optuna study will gather the phone information obtained through adb.exe (Android Developer mode) and define them as metrics to be studied. For example, based on the available hardware resources on the phone, alter the delegation ratio between the hardware for optimal performance development.

In [ ]:
# For colab use

!git clone --branch jtan/dev https://github.com/tanjun8802/Mase_EDGE.git 
%cd /content
!rm -rf Mase_EDGE/mase
%cd Mase_EDGE
!git submodule update --init --recursive
%cd mase/
!pip install -e .

In [ ]:
# Get the dataset (Tiny ImageNet)

%cd /content/Mase_EDGE
!wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
!unzip tiny-imagenet-200.zip

In [ ]:
import optuna
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.utils.prune as prune
import torch.export as exir
import torchvision.transforms as T
import torch.optim as optim
import subprocess
import json
import time
import os, sys
from pathlib import Path
from typing import Dict, Any
from chop import MaseGraph
import chop.passes as passes
from torchvision import datasets

PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from EDGE_device.device_specifications import get_hardware_for_phone,load_phone_specs

### 1. Load Metrics from Android Phone

If you are trying to optimise and deploy the model on an Android phone, please plug in the phone to the laptop and make sure the developer mode is on. In settings make sure debugging mode is turned on. Run the following code cell to extract the hardware information. A JSON file will be created to hold the information and it will be used in the configuration setup.

In [ ]:
hw = get_hardware_for_phone() # Make sure the phone is connected and ADB is set up
print("Optimizing for:", hw["model"])
print(hw["gpu"], hw["cpu_cores"], hw["ram_gb"])

In [ ]:
specs = load_phone_specs('EDGE_device/device_specifications.json')          # uses default PHONE_SPECS_FILE
print("Specs dict:", specs)
print("Keys:", list(specs.keys()))

if specs:
    hw = list(specs.values())[0]
    print(hw["model"], hw["gpu"], hw["cpu_cores"], hw["ram_gb"])
else:
    print("No specs found; JSON file missing or empty.")

Specs dict: {'HONOR:PTP-N49:SM8750': {'manufacturer': 'HONOR', 'model': 'PTP-N49', 'brand': 'HONOR', 'soc_manufacturer': 'Qualcomm', 'soc_model': 'SM8750', 'hardware': 'qcom', 'cpu_cores': 8, 'cpu_freq_max_ghz': 3.2, 'ram_gb': 10.9794921875, 'has_gpu': True, 'gpu': 'Adreno 830', 'gpu_compute_level': 'high', 'has_npu': False, 'npu_type': 'none', 'prefers_npu': False, 'prefers_gpu': True, 'prefers_cpu': False}}
Keys: ['HONOR:PTP-N49:SM8750']
PTP-N49 Adreno 830 8 10.9794921875


### 2. Defining the Search Space

Here the code prepares the configurations for the pipeline

In [ ]:
def edge_optuna_config(trial):
    """Practical config: Pruning + Backend + Quant (no full NAS)."""
    
    # Backend options (hardware‑aware)
    backend_opts = ["cpu"]
    if hw["prefers_gpu"]:
        backend_opts.extend(["xnnpack", "vulkan"])
    if hw["has_npu"]:
        backend_opts.extend(["qnn", "nnapi"])
    
    config = {
        # Compression
        "prune_ratio": trial.suggest_float("prune_ratio", 0.0, 0.7),
        "quant_bits": trial.suggest_categorical("quant_bits", [8, 16]),
        
        # Delegation
        "backend": trial.suggest_categorical("backend", backend_opts),
        #"fusion": trial.suggest_categorical("fusion", ["conv_bn_relu", "conv_bn", "none"]), executorch normally fuse under the hood
    }
    
    # Backend‑aware quant
    if config["backend"] in ["vulkan", "xnnpack"]:
        config["quant_config"] = trial.suggest_categorical("quant_config", ["int8", "fp16"])
    else:
        config["quant_config"] = "int8"
    
    return config


### 3. Model Optimisation

In [ ]:
# Inherit from Clarence's notebook and mase is used here

def edge_optimise_model(config: Dict[str, Any]) -> nn.Module:
    """ (pruning + PTQ) """
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Load + modify base model
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, 200)  # Tiny ImageNet
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    transform = T.Compose([
        T.Resize(224), T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # 1 batch only (Optuna speed)
    train_dataset = datasets.ImageFolder(root="./tiny-imagenet-200/train", transform=transform)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
    
    model.train()
    imgs, labels = next(iter(train_loader))
    imgs, labels = imgs.to(device), labels.to(device)
    optimizer.zero_grad()
    loss = criterion(model(imgs), labels)
    loss.backward()
    optimizer.step()
    
    example_inputs = torch.randn(1, 3, 224, 224).to(device)
    dummy_inputs = {"x": example_inputs}
    
    mg = MaseGraph(model)
    mg, _ = passes.init_metadata_analysis_pass(mg)
    mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_inputs})


    pruning_config = {
        "weight": {
            "sparsity": config["prune_ratio"],
            "method": "l1-norm",
            "scope": "local",
        },
        "activation": {
            "sparsity": config["prune_ratio"],
            "method": "l1-norm",
            "scope": "local",
        },
    }

    mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)
    
    # Configurable quantization (Optuna param!)
    quantization_config = {
        "by": "type",
        "default": {"config": {"name": None}},
        "linear": {
            "config": {
                "name": "integer",
                "data_in_width": config["quant_bits"],
                "data_in_frac_width": config["quant_bits"] - 1,
                "weight_width": config["quant_bits"],
                "weight_frac_width": config["quant_bits"] - 1,
                "bias_width": 32,
                "bias_frac_width": 0,
            }
        },
    }
    
    mg, _ = passes.quantize_transform_pass(mg, pass_args=quantization_config)
    
    # Return FX GraphModule (ExecuTorch‑ready)
    ptq_model = torch.fx.GraphModule(mg.model, mg.fx_graph).cpu()
    return ptq_model



### 4. Export .pte file for Edge Device

In [ ]:
def edge_export_model(model: nn.Module, trial_id: int, config: Dict[str, Any]) -> str:
    """Export with backend partitioning + fusion."""
    
    example_input = torch.randn(1, 3, 224, 224)
    exported = torch.export.export(model, (example_input,))
    
    # Backend partitioning
    backend_map = {
        "vulkan": "vulkan_partitioner",
        "xnnpack": "xnnpack_partitioner", 
        "cpu": "cpu_partitioner",
        "qnn": "qnn_partitioner",
        "nnapi": "nnapi_partitioner",
    }
    
    # Apply partitioner
    partitioner_name = backend_map.get(config["backend"], "xnnpack_partitioner")
    edge_model = exir.to_backend(partitioner_name, exported)
    
    # Fusion (placeholder)
    if config["fusion"] == "conv_bn_relu":
        # edge_model = fuse_conv_bn_relu(edge_model)  # Implement later
        pass
    
    # Save
    PTE_DIR = Path("pte_files")
    PTE_DIR.mkdir(exist_ok=True)
    pte_path = PTE_DIR / f"resnet18_trial_{trial_id}.pte"
    edge_model.save_graph_to_file(str(pte_path))
    
    return str(pte_path)

### Benchmark for the trials on the phone

The function that pushes the optimised model to run on the phone, and reports back the metrics for evaluation and feedback for optuna study.

The pipeline is: Build & install the Android benchmark app once (via Android Studio or adb install).

Per trial, Python:

- adb push the new .pte

- adb shell am start ... to trigger your app’s Activity/Service

- Waits for the app to run the model and log metrics

In [ ]:
# Allow us to quatify the performance, and also act as the "oracle" for Optuna (returning the metrics)

def edge_benchmark_trial(trial_id: int, pte_path: str) -> Dict[str, float]:
    """Push → benchmark → metrics."""
    try:
        # Push model
        subprocess.run(["adb", "push", pte_path, "/sdcard/resnet18_trial.pte"], 
                      check=True, timeout=10)
        
        # Launch benchmark app
        subprocess.run([
            "adb", "shell", "am", "start",
            "-a", "com.yourapp.BENCHMARK",
            "-e", "trial_id", str(trial_id),
            "-e", "model_path", "/sdcard/resnet18_trial.pte"
        ], check=True, timeout=5)
        
        # Wait for metrics
        return edge_poll_metrics(trial_id)
        
    except Exception as e:
        print(f"Trial {trial_id} failed: {e}")
        return {"top1_acc": 0.0, "latency_p95_ms": 999.0}

def edge_poll_metrics(trial_id: int, timeout: int = 60) -> Dict[str, float]:
    """Poll logcat for JSON metrics."""
    start = time.time()
    while time.time() - start < timeout:
        logcat = subprocess.run([
            "adb", "logcat", "-d", f"Trial{trial_id}:*"
        ], capture_output=True, text=True, timeout=5).stdout
        
        for line in logcat.splitlines():
            if f"Trial {trial_id} METRICS:" in line:
                try:
                    metrics_str = line.split("METRICS:")[1].strip()
                    return json.loads(metrics_str)
                except:
                    continue
        time.sleep(2)
    
    return {"top1_acc": 0.0, "latency_p95_ms": 999.0}


### 5. Defining the Objective Function

In [ ]:
def objective(trial):
    """Full pipeline: config, model, export, phone, score."""
    
    config = edge_optuna_config(trial)
    print(f"Trial {trial.number}: backend={config['backend']}, "
          f"prune={config['prune_ratio']:.1f}, quant={config['quant_config']}")
    
    # Optimize model
    model = edge_optimise_model(config)
    
    # Export for edge device
    pte_path = edge_export_model(model, trial.number, config)
    
    # Phone benchmark (where the pte file is run on the phone and metrics are collected)
    metrics = edge_benchmark_trial(trial.number, pte_path)
    
    # Multi‑objective score
    acc = metrics.get("top1_acc", 0.0)
    latency = metrics.get("latency_p95_ms", 999.0)
    memory = metrics.get("memory_peak_mb", 9999.0)
    
    # Score to maximise accuracy, but penalise latency and memory
    score = acc - 0.02 * (latency / 30.0) - 0.001 * (memory / 2000.0) # just an example need to review this
    
    trial.set_user_attr("top1_acc", acc)
    trial.set_user_attr("latency_ms", latency)
    trial.set_user_attr("memory_mb", memory)
    for k, v in config.items():
        trial.set_user_attr(k, v)
    
    print(f"  → acc={acc:.3f}, lat={latency:.1f}ms, score={score:.3f}")
    return score

### 6. Optuna Study

In [ ]:
# Create + run study
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="resnet18_edge_opt",
    storage="sqlite:///optuna_resnet18.db",  # Save progress
    load_if_exists=True,
)

# Run 50 trials
study.optimize(objective, n_trials=50)

print(f"\nBest trial #{study.best_trial.number}")
print(f"   Score: {study.best_value:.3f}")
print("   Config:", study.best_params)

# Quick viz

optuna.visualization.plot_optimization_history(study).show()


### Extract the best model

In [ ]:
# Get best config
best_trial = study.best_trial
best_config = {k: v for k, v in best_trial.user_attrs.items() 
               if k in ["prune_ratio", "quant_bits", "backend", "fusion"]}

# Rebuild + export best model
best_model = edge_optimise_model(best_config)
final_pte = edge_export_model(best_model, 999, best_config)

print(f"Best model: {final_pte}")
print("Deploy with: adb push", final_pte, "/sdcard/resnet18_best.pte")
